# 00 – Full Dataset Preprocessing

Complete end-to-end preprocessing pipeline: raw JSON → Parquet → global ID mappings → CSR matrix.

This is a **one-time** preprocessing step that creates the foundation for all downstream analysis:
1. **JSON → Parquet** conversion (if needed) 
   - `business/state=*/part-*.parquet` - Businesses partitioned by state
   - `review/year=*/part-*.parquet` - Reviews partitioned by year
2. **Build global ID mappings**
   - `user2index.pkl` - Maps user_id → consistent integer index
   - `item2index.pkl` - Maps business_id → consistent integer index
3. **Create CSR matrices**
   - `R_full.npz` - Full CSR user-item interaction matrix (all states)
   - `R_{STATE}_compact.npz` - Per-state CSR variants (compact versions)
   - `state_statistics.csv` - Statistics by state

**Why separate preprocessing?** Consistent indices across experiments enable reproducible neuron labeling and interpretability. Filtering decisions are deferred to training time.

## Setup

In [1]:
import sys
from pathlib import Path
import pickle
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.sparse import save_npz, csr_matrix, lil_matrix

# Handle imports from src
try:
    from src.utils import Config, load_config
    from src.data.yelp_loader import load_reviews, load_businesses
    from src.data.preprocessing import build_csr
    print("✓ Direct imports successful (src is in Python path)")
except ImportError:
    # Try adding ../src to path (notebook is in notebooks/ subdirectory)
    parent = Path(".").resolve().parent
    src_path = parent / "src"
    if src_path.exists():
        sys.path.insert(0, str(src_path.parent))
        from src.utils import Config, load_config
        from src.data.yelp_loader import load_reviews, load_businesses
        from src.data.preprocessing import build_csr
        print(f"✓ Imports successful (added {src_path.parent} to path)")
    else:
        print(f"⚠ Could not find src/ directory at {src_path}")
        print("   Make sure to run this notebook from the project root")

✓ Imports successful (added C:\Users\elisk\Desktop\2024-25\Diplomka\Github\Diplomov-pr-ce to path)


## Load Configuration

## (Optional) Convert Raw JSON to Parquet

Converts raw Yelp JSON files to partitioned Parquet format for faster loading.

**When does this run?**
- Automatically: If Parquet files don't exist at the path in `configs/default.yaml`
- Skipped: If Parquet files already exist (fast-checks for existing data)

**Output structure:**
- `business/state=XX/part-0.parquet` - Businesses partitioned by state (AB, AZ, CA, etc.)
- `review/year=YYYY/part-0.parquet` - Reviews partitioned by year (2005, 2006, ..., 2021)

**Why partitioning?**
- **State partitioning**: Enables efficient filtering for per-state analysis (used by training notebook)
- **Year partitioning**: Enables temporal analysis and efficient date range queries
- **Compression**: Uses Snappy compression to reduce file sizes

**Approach:**
1. Reads JSON as line-delimited text
2. Converts to pandas DataFrame using `pd.read_json(..., lines=True)`
3. Partitions and saves as Parquet using `to_parquet(... compression='snappy')`

This mirrors the preprocessing principle: work with DataFrames, apply transformations, persist efficiently.

In [2]:
# Find project root
project_root = Path(".").resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

config_path = project_root / "configs" / "default.yaml"

try:
    config = load_config(str(config_path))
    print(f"✓ Config loaded from: {config_path}")
except Exception as e:
    print(f"⚠ Error loading config: {e}")
    print(f"   Looked for: {config_path}")

# Data paths
parquet_dir_str = config['data']['parquet_dir']
PARQUET_DIR = Path(parquet_dir_str) if Path(parquet_dir_str).is_absolute() else project_root / parquet_dir_str

# Output directory for preprocessed data
PREPROCESS_DIR = project_root / "data" / "preprocessed_yelp"
PREPROCESS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Parquet directory: {PARQUET_DIR}")
print(f"✓ Preprocessed data will be saved to: {PREPROCESS_DIR}")

✓ Config loaded from: C:\Users\elisk\Desktop\2024-25\Diplomka\Github\Diplomov-pr-ce\configs\default.yaml

✓ Parquet directory: C:\Users\elisk\Desktop\2024-25\Diplomka\Github\Diplomov-pr-ce\..\..\Yelp-JSON\yelp_parquet
✓ Preprocessed data will be saved to: C:\Users\elisk\Desktop\2024-25\Diplomka\Github\Diplomov-pr-ce\data\preprocessed_yelp


In [3]:
# Check if Parquet files already exist
parquet_business_dir = PARQUET_DIR / "business"
parquet_review_dir = PARQUET_DIR / "review"

if parquet_business_dir.exists() and parquet_review_dir.exists():
    # Check if they have files
    biz_files = list(parquet_business_dir.glob("state=*/*.parquet"))
    rev_files = list(parquet_review_dir.glob("year=*/*.parquet"))
    
    if biz_files and rev_files:
        print(f"✓ Parquet files already exist at {PARQUET_DIR}")
        print(f"  Business: {len(biz_files)} files")
        print(f"  Reviews: {len(rev_files)} files")
        print(f"  → Skipping JSON conversion")
        parquet_exists = True
    else:
        parquet_exists = False
else:
    parquet_exists = False

if not parquet_exists:
    # Convert JSON to Parquet
    print(f"\n{'='*80}")
    print("CONVERTING YELP JSON TO PARQUET")
    print(f"{'='*80}")
    
    # Find raw JSON directory - check multiple possible locations
    possible_json_dirs = [
        project_root.parent / "Yelp-JSON" / "yelp_dataset",
        project_root.parent.parent / "Yelp-JSON" / "yelp_dataset",
        Path("../../Yelp-JSON/yelp_dataset"),
    ]
    
    JSON_DIR = None
    for path in possible_json_dirs:
        if path.exists():
            JSON_DIR = path.resolve()
            break
    
    if JSON_DIR is None:
        print(f"\n⚠ JSON directory not found. Checked:")
        for path in possible_json_dirs:
            print(f"  • {path}")
        print(f"\nIf you have raw Yelp JSON files, place them in one of these locations.")
        print(f"Then re-run this cell to start conversion.")
    else:
        print(f"\nJSON source: {JSON_DIR}")
        
        # 1. Convert businesses JSON to Parquet (partitioned by state)
        print("\n1. Converting businesses.json → Parquet (partitioned by state)...")
        businesses_json = JSON_DIR / "yelp_academic_dataset_business.json"
        
        if businesses_json.exists():
            # Read as DataFrame (line-delimited JSON)
            biz_df = pd.read_json(businesses_json, lines=True)
            print(f"   Loaded {len(biz_df):,} businesses")
            
            # Create Parquet partitioned directory structure
            parquet_business_dir.mkdir(parents=True, exist_ok=True)
            
            # Write partitioned by state
            for state in sorted(biz_df['state'].unique()):
                state_data = biz_df[biz_df['state'] == state]
                state_dir = parquet_business_dir / f"state={state}"
                state_dir.mkdir(parents=True, exist_ok=True)
                
                parquet_file = state_dir / f"part-0.parquet"
                state_data.to_parquet(parquet_file, compression='snappy', index=False)
                print(f"   ✓ {state}: {len(state_data):,} businesses → {parquet_file.name}")
        else:
            print(f"   ⚠ Not found: {businesses_json}")
        
        # 2. Convert reviews JSON to Parquet (partitioned by year)
        print("\n2. Converting reviews.json → Parquet (partitioned by year)...")
        reviews_json = JSON_DIR / "yelp_academic_dataset_review.json"
        
        if reviews_json.exists():
            # Read as DataFrame (line-delimited JSON)
            rev_df = pd.read_json(reviews_json, lines=True)
            print(f"   Loaded {len(rev_df):,} reviews")
            
            # Extract year from date
            rev_df['date'] = pd.to_datetime(rev_df['date'])
            rev_df['year'] = rev_df['date'].dt.year
            
            # Create Parquet partitioned directory structure
            parquet_review_dir.mkdir(parents=True, exist_ok=True)
            
            # Write partitioned by year
            for year in sorted(rev_df['year'].unique()):
                year_data = rev_df[rev_df['year'] == year].drop(columns=['year'])
                year_dir = parquet_review_dir / f"year={int(year)}"
                year_dir.mkdir(parents=True, exist_ok=True)
                
                parquet_file = year_dir / f"part-0.parquet"
                year_data.to_parquet(parquet_file, compression='snappy', index=False)
                print(f"   ✓ {int(year)}: {len(year_data):,} reviews → {parquet_file.name}")
        else:
            print(f"   ⚠ Not found: {reviews_json}")
        
        print(f"\n✓ Parquet conversion complete!")
        print(f"  Saved to: {PARQUET_DIR}")

✓ Parquet files already exist at C:\Users\elisk\Desktop\2024-25\Diplomka\Github\Diplomov-pr-ce\..\..\Yelp-JSON\yelp_parquet
  Business: 27 files
  Reviews: 18 files
  → Skipping JSON conversion


## Load Full Dataset (All States)

In [4]:
print("="*80)
print("LOADING FULL YELP DATASET")
print("="*80)

# Load reviews from Parquet - NO state filter (get everything)
print("\nLoading reviews from all states...")
reviews = load_reviews(
    PARQUET_DIR,
    pos_threshold=config['data']['pos_threshold'],
    year_min=config['data'].get('year_min'),
    year_max=config['data'].get('year_max'),
)

print(f"✓ Loaded {len(reviews):,} reviews")
print(f"  Columns: {list(reviews.columns)}")
print(f"  User-item pairs: {reviews[['user_id', 'business_id']].drop_duplicates().shape[0]:,}")

# Show sample
print(f"\nSample reviews:")
print(reviews.head())

LOADING FULL YELP DATASET

Loading reviews from all states...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Loaded 4,684,545 reviews
  Columns: ['user_id', 'business_id', 'ts', 'implicit']
  User-item pairs: 4,558,128

Sample reviews:
                  user_id             business_id             ts  implicit
0  c8qFkI_VusWo0xZvkjfBWQ  PY9GRfzr4nTZeINf346QOw  1116117569000         1
1  bA6sFAZTIc5ddwpnB_WJUw  GgcvRnt5_z3NEC0D6vNncQ  1126757632000         1
2  DOFamdMaC_hjn9zg3Gs-0w  I13c2-Yo7hn2BKwCI81HSQ  1115784994000         1
3  iUeZhkI0OK0BisakOkb3pQ  j8JOZvfeHEfUWq3gEz6ABQ  1115819337000         1
4  n-lBS02-3yvlY5Q91mmwDA  ompDR5sUDpoI6gnTldmneQ  1121013965000         1


In [5]:
# Load ALL businesses (no state filter, no review count filter)
# We keep all businesses in preprocessing so every review has a match
# Any filtering (business quality, review counts, etc.) should happen during training
print("\nLoading businesses from all states...")
businesses = load_businesses(
    PARQUET_DIR,
    state_filter=None,  # NO state filtering
    min_review_count=0,  # NO business-level filtering - keep everything
)

print(f"✓ Loaded {len(businesses):,} businesses")
print(f"  States represented: {businesses['state'].nunique()}")
print(f"  State distribution:\n{businesses['state'].value_counts().head(10)}")


Loading businesses from all states...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Loaded 150,346 businesses
  States represented: 27
  State distribution:
state
PA    34039
FL    26330
TN    12056
IN    11247
MO    10913
LA     9924
AZ     9912
NJ     8536
NV     7715
AB     5573
Name: count, dtype: int64


In [6]:
# Sanity check: filter reviews to only include businesses in our dataset
# (Should be no-op since we load all businesses, but catch any data inconsistencies)
print("\nSanity check: matching reviews to businesses...")
business_ids = set(businesses["business_id"].values)
reviews_before = len(reviews)
reviews = reviews[reviews["business_id"].isin(business_ids)]
reviews_after = len(reviews)

if reviews_before == reviews_after:
    print(f"✓ All {reviews_after:,} reviews have matching businesses")
else:
    print(f"⚠ Removed {reviews_before - reviews_after:,} orphaned reviews ({100*(reviews_before-reviews_after)/reviews_before:.2f}%)")
    print(f"  (These reviews reference businesses not in the dataset)")
print(f"  Unique user-item pairs: {reviews[['user_id', 'business_id']].drop_duplicates().shape[0]:,}")


Sanity check: matching reviews to businesses...


✓ All 4,684,545 reviews have matching businesses
  Unique user-item pairs: 4,558,128


In [7]:
# DEBUG: Check what columns exist in reviews
print("\n" + "="*80)
print("DEBUG: Reviews DataFrame structure")
print("="*80)
print(f"\nColumns in reviews: {list(reviews.columns)}")
print(f"Shape: {reviews.shape}")
print(f"\nFirst row:\n{reviews.iloc[0]}")
print(f"\nData types:\n{reviews.dtypes}")



DEBUG: Reviews DataFrame structure

Columns in reviews: ['user_id', 'business_id', 'ts', 'implicit']
Shape: (4684545, 4)

First row:
user_id        c8qFkI_VusWo0xZvkjfBWQ
business_id    PY9GRfzr4nTZeINf346QOw
ts                      1116117569000
implicit                            1
Name: 0, dtype: object

Data types:
user_id        object
business_id    object
ts              int64
implicit        int32
dtype: object


## Build CSR Matrix & ID Mappings

In [8]:
print("\n" + "="*80)
print("BUILDING CSR MATRIX WITH CONSISTENT ID MAPPINGS")
print("="*80)

# Build CSR matrix from reviews
print("\nBuilding CSR...")
dataset = build_csr(reviews)

X_full_csr = dataset.csr
user_map = dataset.user_map
item_map = dataset.item_map

print(f"\n✓ CSR Matrix Built:")
print(f"  Users: {X_full_csr.shape[0]:,}")
print(f"  Items: {X_full_csr.shape[1]:,}")
print(f"  Interactions: {X_full_csr.nnz:,}")
print(f"  Sparsity: {(1 - X_full_csr.nnz / (X_full_csr.shape[0] * X_full_csr.shape[1])) * 100:.2f}%")

print(f"\n✓ ID Mappings:")
print(f"  User IDs in map: {len(user_map):,}")
print(f"  Item IDs in map: {len(item_map):,}")

# Sanity check
assert len(user_map) == X_full_csr.shape[0], "User map size mismatch!"
assert len(item_map) == X_full_csr.shape[1], "Item map size mismatch!"
print(f"  ✓ Maps match CSR dimensions")


BUILDING CSR MATRIX WITH CONSISTENT ID MAPPINGS

Building CSR...

✓ CSR Matrix Built:
  Users: 1,464,850
  Items: 147,491
  Interactions: 4,558,128
  Sparsity: 100.00%

✓ ID Mappings:
  User IDs in map: 1,464,850
  Item IDs in map: 147,491
  ✓ Maps match CSR dimensions


## Save Preprocessed Data

In [9]:
print("\n" + "="*80)
print("SAVING PREPROCESSED DATA")
print("="*80)

# Save user mapping
user_map_path = PREPROCESS_DIR / "user2index.pkl"
with open(user_map_path, "wb") as f:
    pickle.dump(user_map, f)
print(f"\n✓ Saved user mapping: {user_map_path}")
print(f"  {len(user_map):,} unique users")

# Save item mapping
item_map_path = PREPROCESS_DIR / "item2index.pkl"
with open(item_map_path, "wb") as f:
    pickle.dump(item_map, f)
print(f"\n✓ Saved item mapping: {item_map_path}")
print(f"  {len(item_map):,} unique businesses")

# Save CSR matrix
csr_path = PREPROCESS_DIR / "R_full.npz"
save_npz(csr_path, X_full_csr)
print(f"\n✓ Saved CSR matrix: {csr_path}")
print(f"  Shape: {X_full_csr.shape[0]:,} × {X_full_csr.shape[1]:,}")
print(f"  Interactions: {X_full_csr.nnz:,}")
print(f"  File size: {csr_path.stat().st_size / 1024 / 1024:.1f} MB")


SAVING PREPROCESSED DATA



✓ Saved user mapping: C:\Users\elisk\Desktop\2024-25\Diplomka\Github\Diplomov-pr-ce\data\preprocessed_yelp\user2index.pkl
  1,464,850 unique users

✓ Saved item mapping: C:\Users\elisk\Desktop\2024-25\Diplomka\Github\Diplomov-pr-ce\data\preprocessed_yelp\item2index.pkl
  147,491 unique businesses

✓ Saved CSR matrix: C:\Users\elisk\Desktop\2024-25\Diplomka\Github\Diplomov-pr-ce\data\preprocessed_yelp\R_full.npz
  Shape: 1,464,850 × 147,491
  Interactions: 4,558,128
  File size: 13.1 MB


In [10]:
# Save metadata about the preprocessing
metadata = {
    "timestamp": datetime.now().isoformat(),
    "n_users": int(X_full_csr.shape[0]),
    "n_items": int(X_full_csr.shape[1]),
    "n_interactions": int(X_full_csr.nnz),
    "sparsity_pct": float(100 * (1 - X_full_csr.nnz / (X_full_csr.shape[0] * X_full_csr.shape[1]))),
    "pos_threshold": config['data']['pos_threshold'],
    "year_min": config['data'].get('year_min'),
    "year_max": config['data'].get('year_max'),
    "min_business_reviews": config['data'].get('min_review_count', 5),
    "config_source": str(config_path),
}

import json
metadata_path = PREPROCESS_DIR / "metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\n✓ Saved metadata: {metadata_path}")
print(f"  {json.dumps(metadata, indent=2)}")


✓ Saved metadata: C:\Users\elisk\Desktop\2024-25\Diplomka\Github\Diplomov-pr-ce\data\preprocessed_yelp\metadata.json
  {
  "timestamp": "2026-04-04T19:45:13.458593",
  "n_users": 1464850,
  "n_items": 147491,
  "n_interactions": 4558128,
  "sparsity_pct": 99.99789026532362,
  "pos_threshold": 4.0,
  "year_min": null,
  "year_max": null,
  "min_business_reviews": 20,
  "config_source": "C:\\Users\\elisk\\Desktop\\2024-25\\Diplomka\\Github\\Diplomov-pr-ce\\configs\\default.yaml"
}


## Save Per-State Variants

In [11]:
print("\n" + "="*80)
print("SAVING PER-STATE VARIANTS")
print("="*80)

# Get unique states from businesses
states = sorted(businesses['state'].unique())
print(f"\nFound {len(states)} states: {', '.join(states)}")

# For each state, create a filtered CSR matrix
state_stats = []

for state in states:
    # Get business IDs for this state
    state_businesses = businesses[businesses['state'] == state]
    state_business_ids = set(state_businesses["business_id"].values)
    
    # Filter reviews to this state
    state_reviews = reviews[reviews["business_id"].isin(state_business_ids)]
    
    # Build using lil_matrix (more efficient for incremental construction)
    state_csr = lil_matrix((X_full_csr.shape[0], X_full_csr.shape[1]), dtype=np.float32)
    
    # Map state reviews using the global mappings
    if len(state_reviews) > 0:
        for _, row in state_reviews.iterrows():
            user_idx = user_map.get(row['user_id'])
            item_idx = item_map.get(row['business_id'])
            if user_idx is not None and item_idx is not None:
                state_csr[user_idx, item_idx] = 1.0
    
    # Convert to CSR for storage
    state_csr = state_csr.tocsr()
    
    # Save state-specific CSR
    state_file = PREPROCESS_DIR / f"R_{state}.npz"
    save_npz(state_file, state_csr)
    
    # Remove all-zero rows and columns for space efficiency
    state_csr_compact = state_csr.copy()
    row_nnz = np.diff(state_csr_compact.indptr)
    col_nnz = np.diff(state_csr_compact.tocsc().indptr)
    keep_rows = row_nnz > 0
    keep_cols = col_nnz > 0
    state_csr_compact = state_csr_compact[keep_rows][:, keep_cols]
    
    state_file_compact = PREPROCESS_DIR / f"R_{state}_compact.npz"
    save_npz(state_file_compact, state_csr_compact)
    
    state_stats.append({
        'state': state,
        'n_users_full': state_csr.shape[0],
        'n_items_full': state_csr.shape[1],
        'n_interactions': state_csr.nnz,
        'n_users_active': state_csr_compact.shape[0],
        'n_items_active': state_csr_compact.shape[1],
        'file_full_mb': state_file.stat().st_size / 1024 / 1024,
        'file_compact_mb': state_file_compact.stat().st_size / 1024 / 1024,
    })
    
    print(f"\n✓ {state}: {state_csr.nnz:,} interactions")
    print(f"    users: {keep_rows.sum():,}/{X_full_csr.shape[0]:,}, items: {keep_cols.sum():,}/{X_full_csr.shape[1]:,}")
    print(f"    files: {state_file.name} ({state_stats[-1]['file_full_mb']:.1f}MB), {state_file_compact.name} ({state_stats[-1]['file_compact_mb']:.1f}MB)")

# Save state statistics
state_stats_df = pd.DataFrame(state_stats)
stats_csv_path = PREPROCESS_DIR / "state_statistics.csv"
state_stats_df.to_csv(stats_csv_path, index=False)
print(f"\n✓ Saved state statistics: {stats_csv_path}")
print(state_stats_df.to_string(index=False))


SAVING PER-STATE VARIANTS

Found 27 states: AB, AZ, CA, CO, DE, FL, HI, ID, IL, IN, LA, MA, MI, MO, MT, NC, NJ, NV, PA, SD, TN, TX, UT, VI, VT, WA, XMS

✓ AB: 66,145 interactions
    users: 15,438/1,464,850, items: 5,389/147,491
    files: R_AB.npz (0.2MB), R_AB_compact.npz (0.1MB)

✓ AZ: 273,007 interactions
    users: 92,561/1,464,850, items: 9,765/147,491
    files: R_AZ.npz (0.8MB), R_AZ_compact.npz (0.6MB)

✓ CA: 248,516 interactions
    users: 123,107/1,464,850, items: 5,171/147,491
    files: R_CA.npz (0.7MB), R_CA_compact.npz (0.6MB)

✓ CO: 20 interactions
    users: 20/1,464,850, items: 3/147,491
    files: R_CO.npz (0.0MB), R_CO_compact.npz (0.0MB)

✓ DE: 40,962 interactions
    users: 21,353/1,464,850, items: 2,180/147,491
    files: R_DE.npz (0.1MB), R_DE_compact.npz (0.1MB)

✓ FL: 761,775 interactions
    users: 257,746/1,464,850, items: 25,836/147,491
    files: R_FL.npz (2.2MB), R_FL_compact.npz (1.9MB)

✓ HI: 26 interactions
    users: 26/1,464,850, items: 2/147,491
  

In [12]:
print("\n" + "="*80)
print("PREPROCESSING COMPLETE ✓")
print("="*80)

print(f"\nGenerated files in: {PREPROCESS_DIR}")
print(f"  • user2index.pkl ({len(user_map):,} users)")
print(f"  • item2index.pkl ({len(item_map):,} items)")
print(f"  • R_full.npz ({X_full_csr.shape[0]:,} × {X_full_csr.shape[1]:,}, {X_full_csr.nnz:,} interactions)")
print(f"  • metadata.json")


PREPROCESSING COMPLETE ✓

Generated files in: C:\Users\elisk\Desktop\2024-25\Diplomka\Github\Diplomov-pr-ce\data\preprocessed_yelp
  • user2index.pkl (1,464,850 users)
  • item2index.pkl (147,491 items)
  • R_full.npz (1,464,850 × 147,491, 4,558,128 interactions)
  • metadata.json
